In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from tqdm import tqdm
from livelossplot import PlotLossesKeras

I0000 00:00:1778757616.283127  137856 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv("Data_Clean.csv")

np.random.seed(42)
jumlah_suntikan = 5000 

target_asli = np.random.uniform(1_000_000, 2_000_000_000, jumlah_suntikan)
nabung_asli = np.random.uniform(250_000, 5_000_000, jumlah_suntikan)

terkumpul_asli = np.where(
    np.random.rand(jumlah_suntikan) > 0.5,
    0,
    target_asli * np.random.uniform(0.1, 0.9, jumlah_suntikan)
)

estimasi_ekstrem = np.ceil((target_asli - terkumpul_asli) / nabung_asli)

df_ekstrem = pd.DataFrame({
    'total_terkumpul': np.log1p(terkumpul_asli),
    'target_nominal': np.log1p(target_asli),
    'nominal_nabung': np.log1p(nabung_asli),
    'estimasi_sisa_nabung': estimasi_ekstrem
})

df_augmented = pd.concat([df, df_ekstrem], ignore_index=True)

X = df_augmented[['total_terkumpul', 'target_nominal', 'nominal_nabung']]
y = df_augmented['estimasi_sisa_nabung']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Jumlah data asli: {len(df)}")
print(f"Jumlah data setelah ditambah kasus ekstrem: {len(df_augmented)}")

Jumlah data asli: 29040
Jumlah data setelah ditambah kasus ekstrem: 34040


In [3]:
scaler_x = StandardScaler() 
X_train_scaled = scaler_x.fit_transform(X_train)
X_test_scaled = scaler_x.transform(X_test)

y_train_scaled = np.log1p(y_train.values).reshape(-1, 1)
y_test_scaled = np.log1p(y_test.values).reshape(-1, 1)

In [4]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

In [5]:
print("=" * 50)
print("  Training Deep Learning (Functional API & Huber Loss)  ")
print("=" * 50)

class CustomTrainingLogger(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 50 == 0:
            print(f"✅ Epoch {epoch+1:03d}/500 | Loss (Huber): {logs.get('loss'):.6f} | Val MAE: {logs.get('val_mae'):.6f}")

inputs = layers.Input(shape=(X_train_scaled.shape[1],), name="input_features")

x = layers.Dense(512, activation='relu')(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)

outputs = layers.Dense(1, name="estimasi_output")(x)
dl_model = keras.Model(inputs=inputs, outputs=outputs)

dl_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), 
                 loss=keras.losses.Huber(), 
                 metrics=['mae'])

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=60, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=15, min_lr=0.000001)

history = dl_model.fit(
    X_train_scaled, y_train_scaled,
    epochs=500, batch_size=64, validation_split=0.2,
    callbacks=[early_stop, reduce_lr, CustomTrainingLogger()], 
    verbose=0 # Matikan spam teks bawaan
)

dl_preds_scaled = dl_model.predict(X_test_scaled, verbose=0)
dl_preds = np.expm1(dl_preds_scaled).flatten()

  Training Deep Learning (Functional API & Huber Loss)  


W0000 00:00:1778757620.784852  137856 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


✅ Epoch 050/500 | Loss (Huber): 0.008853 | Val MAE: 0.051647
✅ Epoch 100/500 | Loss (Huber): 0.004920 | Val MAE: 0.039270
✅ Epoch 150/500 | Loss (Huber): 0.002703 | Val MAE: 0.027866
✅ Epoch 200/500 | Loss (Huber): 0.001401 | Val MAE: 0.019851
✅ Epoch 250/500 | Loss (Huber): 0.001290 | Val MAE: 0.017370


In [6]:
def evaluate(name, true, pred):
    print(f"\n--- {name} ---")
    print(f"MAE  : {mean_absolute_error(true, pred):.4f}")
    print(f"R2   : {r2_score(true, pred):.4f}")

evaluate("Random Forest", y_test, rf_preds)
evaluate("Deep Learning", y_test, dl_preds)


--- Random Forest ---
MAE  : 7.0369
R2   : 0.9964

--- Deep Learning ---
MAE  : 2.0108
R2   : 0.9996


In [7]:
model_filename = "model_tabungan_production.keras"
dl_model.save(model_filename)
print(f"💾 Model berhasil disimpan sebagai: {model_filename}")

scaler_filename = "scaler_x_tabungan.pkl"
joblib.dump(scaler_x, scaler_filename)
print(f"💾 Scaler X berhasil disimpan sebagai: {scaler_filename}")

💾 Model berhasil disimpan sebagai: model_tabungan_production.keras
💾 Scaler X berhasil disimpan sebagai: scaler_x_tabungan.pkl


In [11]:
print("\n" + "="*70)
print("  UJI COBA PREDIKSI PRODUKSI (BATCH INFERENCE)")
print("="*70)

loaded_model = keras.models.load_model(model_filename)
loaded_scaler = joblib.load(scaler_filename)

data_uji = [
    {'skenario': 'Mulai dari 0', 'terkumpul': 0, 'target': 10_000_000, 'nabung': 200_000},
    {'skenario': 'Target Sultan', 'terkumpul': 0, 'target': 1_000_000_000, 'nabung': 2_500_000},
    {'skenario': 'Normal', 'terkumpul': 500_000, 'target': 2_000_000, 'nabung': 100_000}
]

df_uji = pd.DataFrame(data_uji)

input_data = pd.DataFrame({
    'total_terkumpul': np.log1p(df_uji['terkumpul']),
    'target_nominal': np.log1p(df_uji['target']),
    'nominal_nabung': np.log1p(df_uji['nabung'])
})

input_scaled = loaded_scaler.transform(input_data)

res_scaled = loaded_model.predict(input_scaled, verbose=0)
df_uji['prediksi_dl'] = np.expm1(res_scaled).flatten()
df_uji['rumus_asli'] = (df_uji['target'] - df_uji['terkumpul']) / df_uji['nabung']

print(df_uji[['skenario', 'rumus_asli', 'prediksi_dl']].to_string(index=False))


  UJI COBA PREDIKSI PRODUKSI (BATCH INFERENCE)
     skenario  rumus_asli  prediksi_dl
 Mulai dari 0        50.0    52.111439
Target Sultan       400.0   398.806488
       Normal        15.0    15.831597
